# Quant Trading Research Platform — WalkthroughEnd-to-end demo of the pipeline: data -> features -> signal -> backtest -> costs -> risk -> performance.Run `streamlit run src/dashboard.py` for the interactive version. This notebook exercises the same engine underneath, so you can inspect intermediate outputs step by step.

In [ ]:
import syssys.path.insert(0, '../src')import pandas as pdimport matplotlib.pyplot as pltfrom data_loader import load_universefrom feature_engineering import build_feature_matrixfrom signals import STRATEGY_REGISTRY, multi_factor_signalfrom backtest_engine import run_backtest, BacktestConfigfrom metrics import full_tearsheet, rolling_sharpe

## 1. Load dataSynthetic fallback keeps this notebook runnable offline; swap `force_synthetic=False` with `yfinance` installed for live data.

In [ ]:
price = load_universe(['RELIANCE.NS'], '2018-01-01', '2024-12-31', force_synthetic=True)['RELIANCE.NS']bench = load_universe(['NIFTY_INDEX'], '2018-01-01', '2024-12-31', force_synthetic=True)['NIFTY_INDEX']price['close'].plot(title='RELIANCE.NS (synthetic demo series)', figsize=(10,4))

## 2. Feature engineering

In [ ]:
feat = build_feature_matrix(price, benchmark_df=bench)feat[['close','sma_20','sma_50','rsi_14']].tail()

## 3. Generate a signalTry swapping `multi_factor_signal` for any strategy in `STRATEGY_REGISTRY`.

In [ ]:
signal, composite = multi_factor_signal(feat)signal.value_counts()

## 4. Backtest -> costs -> risk overlay -> equity curve

In [ ]:
results = run_backtest(feat, signal, BacktestConfig())results['equity_curve'].plot(title='Equity Curve (net of costs)', figsize=(10,4))

## 5. Performance tearsheet

In [ ]:
bench_ret = bench['close'].pct_change()tear = full_tearsheet(results['net_returns'], bench_ret, results['weights'])pd.DataFrame(tear.items(), columns=['Metric','Value'])

## 6. Rolling Sharpe

In [ ]:
rolling_sharpe(results['net_returns']).plot(title='Rolling 63d Sharpe', figsize=(10,4))